# Voice Spoof Detection — WavLM + Wav2Vec2 (Colab)
Akademik PoC. ASVspoof 2019 LA üzerinde iki SSL encoder eğitilir, kalibre edilir ve late fusion ile birleştirilir.

**Önerilen runtime:** GPU (T4 yeterli).

## 1. Bağımlılıklar

In [ ]:
%cd /content
!git clone https://example.com/your/voice-spoof-detection.git || true
%cd voice-spoof-detection
!pip install -q -r requirements.txt

## 2. Google Drive mount (opsiyonel, dataset için)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Dataset yolu
ASVspoof 2019 LA paketinin yolunu ayarla. Dosya yapısı:

```
LA/
  ASVspoof2019_LA_train/flac/...
  ASVspoof2019_LA_dev/flac/...
  ASVspoof2019_LA_eval/flac/...
  ASVspoof2019_LA_cm_protocols/
```

In [ ]:
import os
DATASET_ROOT = '/content/drive/MyDrive/ASVspoof2019/LA'
os.environ['ASVSPOOF_LA_ROOT'] = DATASET_ROOT
assert os.path.isdir(DATASET_ROOT), f'Yol yok: {DATASET_ROOT}'

## 4. Protocol doğrulaması

In [ ]:
from src.data.protocol import load_partition, summarise
for part in ('train', 'dev', 'eval'):
    items = load_partition(DATASET_ROOT, part)
    print(part, summarise(items))

## 5. Hızlı dataset incelemesi

In [ ]:
import random, soundfile as sf, numpy as np
import IPython.display as ipd
samples = random.sample(items, 3)
for it in samples:
    audio, sr = sf.read(it.audio_path)
    print(it.file_id, it.system_id, 'spoof' if it.label==1 else 'bonafide', sr, audio.shape)
    ipd.display(ipd.Audio(audio, rate=sr))

## 6. WavLM eğitimi

In [ ]:
!python -m src.train --config configs/wavlm.yaml --dataset_root $ASVSPOOF_LA_ROOT

## 7. Wav2Vec2 eğitimi

In [ ]:
!python -m src.train --config configs/wav2vec2.yaml --dataset_root $ASVSPOOF_LA_ROOT

## 8-10. Evaluation + Calibration + Fusion

In [ ]:
!python -m src.evaluate \
    --wavlm-checkpoint   checkpoints/wavlm/best.pt \
    --wav2vec2-checkpoint checkpoints/wav2vec2/best.pt \
    --dataset_root $ASVSPOOF_LA_ROOT \
    --fusion

## 11. Grafik ve karşılaştırma tabloları

In [ ]:
import pandas as pd, IPython.display as ipd
ipd.display(pd.read_csv('outputs/evaluation/comparison.csv'))
for path in ['outputs/evaluation/wavlm/confusion_matrix.png',
             'outputs/evaluation/wav2vec2/confusion_matrix.png',
             'outputs/evaluation/fusion/confusion_matrix.png',
             'outputs/evaluation/fusion/roc.png']:
    try:
        ipd.display(ipd.Image(path))
    except Exception:
        pass

## 12. Gradio demo

In [ ]:
!python app.py \
    --wavlm-checkpoint   checkpoints/wavlm/best.pt \
    --wav2vec2-checkpoint checkpoints/wav2vec2/best.pt \
    --fusion-bundle      outputs/evaluation/fusion/fusion_bundle.json \
    --share